In [1]:
import sys
import git
import pathlib

# Set up the PROJ_ROOT variable
PROJ_ROOT_PATH = pathlib.Path(git.Repo('.', search_parent_directories=True).working_tree_dir)
PROJ_ROOT =  str(PROJ_ROOT_PATH)
if PROJ_ROOT not in sys.path:
    sys.path.append(PROJ_ROOT)

# Explicitly add the current notebook's directory
CURRENT_DIR = str(pathlib.Path().absolute())
if CURRENT_DIR not in sys.path:
    sys.path.insert(0, CURRENT_DIR)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors
%matplotlib inline  
# Set the float display format to scientific notation
pd.set_option('display.float_format', '{:.2e}'.format)
# pd.set_option('display.float_format', '{:.2f}'.format)
# pd.set_option('display.precision', 2)

import yaml

In [3]:
from library.fridges import TEMP_STAGES, FRIDGE_LIBRARY
FRIDGE_NAME = "XLD1000SL_v3"
cooling_power_budget = FRIDGE_LIBRARY[FRIDGE_NAME]
temp_stage = "4K"
QUBIT_GROUP_SIZE = 8

COOLING_POWER = cooling_power_budget[temp_stage]
NO_OF_QUBIT_GROUPS = int(10_000/QUBIT_GROUP_SIZE)

In [4]:
fridge = "xld1000sl"

In [5]:
def get_amp_hl(fridge, amp, wire):
    exp = "ff-fc-delft-"+ amp + "_" + wire + "-" + fridge
    fname = PROJ_ROOT_PATH / "notebooks" / "experiments" / fridge / amp / exp / f"{exp}.pkl"
    pqfname =  PROJ_ROOT_PATH / "notebooks" / "experiments" / fridge / amp / exp  / f"PQ_{exp}.pkl"
    df =  pd.read_pickle(fname)

    s = df.loc["4K", "AMP_BIAS"]

    phl = s[("PASSIVE", "IDLE")]
    ahl = s[("AMP", "IDLE")]
    if ("AMP_OHMIC", "IDLE") in s.index:
        ohl = s[("AMP_OHMIC", "IDLE")]
    else:
        ohl = 0.0

    scale = COOLING_POWER / NO_OF_QUBIT_GROUPS / 2
    phl *= scale
    ahl *= scale
    ohl *= scale

    thl = phl + ahl + ohl

    return phl, ahl, ohl, thl

In [15]:
# _SI_PREFIX = {
#     -24: "y", -21: "z", -18: "a", -15: "f", -12: "p", -9: r"{\nano\watt}",
#     -6: r"{\micro\watt}", -3: r"{\milli\watt}", 0: "", 3: "k", 6: "M", 9: "G", 12: "T",
#     15: "P", 18: "E", 21: "Z", 24: "Y",
# }

_SI_PREFIX = {
    -24: "y", -21: "z", -18: "a", -15: "f", -12: "p", -9: "n",
    -6: "u", -3: "m", 0: "", 3: "k", 6: "M", 9: "G", 12: "T",
    15: "P", 18: "E", 21: "Z", 24: "Y",
}

def fmt_eng(x: float, sig: int = 3, unit: str = "") -> str:
    """Engineering notation with SI prefix; exponent divisible by 3."""
    x = float(x)
    if x == 0.0 or not np.isfinite(x):
        return f"{x:g}{unit}"

    exp3 = int(np.floor(np.log10(abs(x)) / 3) * 3)
    exp3 = max(min(exp3, 24), -24)  # clamp to known prefixes
    scaled = x / (10 ** exp3)
    prefix = _SI_PREFIX[exp3]

    # sig significant figures, no trailing clutter
    s = f"{scaled:.2f}".rstrip('0').rstrip('.')
    # return rf"\qty{{{s}}}{prefix}{unit}".strip()
    return f"{s}{prefix}{unit}".strip()

In [20]:
amps  = ["hemt", "hemt-lp", "hemt-ulp", "sisv1", ]
wires = ["cu", "mn", "ybco"]

data = {}
for amp in amps:
    row = {}
    for wire in wires:
        phl, ahl, ohl, thl = get_amp_hl(fridge, amp, wire)

        row[wire] = (
            f"AHL: {fmt_eng(ahl)} \newline "
            f"PHL: {fmt_eng(phl)} \newline "
            f"OHL: {fmt_eng(ohl)} \newline "
            f"THL: {fmt_eng(thl)}"
        )
    data[amp] = row

df_amp = pd.DataFrame.from_dict(data, orient="index")
df_amp.index.name = "Amplifier"
df_amp.columns.name = "Wire"

In [21]:
with pd.option_context('display.max_colwidth', None):
  display(df_amp)

Wire,cu,mn,ybco
Amplifier,,,
hemt,AHL: 7.8m \newline PHL: 25.21m \newline OHL: 191.31n \newline THL: 33.01m,AHL: 7.8m \newline PHL: 116.85u \newline OHL: 469.65u \newline THL: 8.39m,AHL: 7.8m \newline PHL: 11.52u \newline OHL: 0 \newline THL: 7.81m
hemt-lp,AHL: 300u \newline PHL: 25.21m \newline OHL: 10.19n \newline THL: 25.51m,AHL: 300u \newline PHL: 116.85u \newline OHL: 25.01u \newline THL: 441.86u,AHL: 300u \newline PHL: 11.52u \newline OHL: 0 \newline THL: 311.52u
hemt-ulp,AHL: 200u \newline PHL: 50.43m \newline OHL: 7.07n \newline THL: 50.63m,AHL: 200u \newline PHL: 233.71u \newline OHL: 17.37u \newline THL: 451.07u,AHL: 200u \newline PHL: 23.05u \newline OHL: 0 \newline THL: 223.05u
sisv1,AHL: 8.1u \newline PHL: 42.02m \newline OHL: 547.91n \newline THL: 42.03m,AHL: 8.1u \newline PHL: 194.75u \newline OHL: 1.35m \newline THL: 1.55m,AHL: 8.1u \newline PHL: 19.21u \newline OHL: 0 \newline THL: 27.31u
